In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
MYTHS = ['victim_intoxication','clothing', 'perpetrator_intoxication', 'resistance']

In [3]:
analysis = "MYTHS"

# Normality

In [4]:
dataset = "JustDetention"
t1_advice_normality = pd.read_csv(f'{analysis}/T1-Normality-{dataset}-advice.csv')
t1_summary_normality = pd.read_csv(f'{analysis}/T1-Normality-{dataset}-summary.csv')
t1_advice_normality['significant'].value_counts(), t1_summary_normality['significant'].value_counts()

(significant
 False    46
 True      2
 Name: count, dtype: int64,
 significant
 False    48
 Name: count, dtype: int64)

In [5]:
dataset = "SurvivorStories"
t1_advice_normality = pd.read_csv(f'{analysis}/T1-Normality-{dataset}-advice.csv')
t1_summary_normality = pd.read_csv(f'{analysis}/T1-Normality-{dataset}-summary.csv')
t1_advice_normality['significant'].value_counts(), t1_summary_normality['significant'].value_counts()

(significant
 False    48
 Name: count, dtype: int64,
 significant
 False    44
 Name: count, dtype: int64)

## Levene Test

In [6]:
dataset = "JustDetention"
t1_advice_levene = pd.read_csv(f'{analysis}/T1-LeveneTest-{dataset}-advice.csv')
t1_summary_levene = pd.read_csv(f'{analysis}/T1-LeveneTest-{dataset}-summary.csv')
t1_advice_levene['significant'].value_counts(), t1_summary_levene['significant'].value_counts()

(significant
 False    16
 Name: count, dtype: int64,
 significant
 False    16
 Name: count, dtype: int64)

In [7]:
dataset = "SurvivorStories"
t1_advice_levene = pd.read_csv(f'{analysis}/T1-LeveneTest-{dataset}-advice.csv')
t1_summary_levene = pd.read_csv(f'{analysis}/T1-LeveneTest-{dataset}-summary.csv')
t1_advice_levene['significant'].value_counts(), t1_summary_levene['significant'].value_counts()

(significant
 False    16
 Name: count, dtype: int64,
 significant
 False    16
 Name: count, dtype: int64)

## ANOVA

In [8]:
dataset = "JustDetention"
t1_advice_anova = pd.read_csv(f'{analysis}/T1-ANOVA-{dataset}-advice.csv')
t1_summary_anova = pd.read_csv(f'{analysis}/T1-ANOVA-{dataset}-summary.csv')
t1_advice_anova['significant'].value_counts(), t1_summary_anova['significant'].value_counts()

(significant
 False    11
 True      5
 Name: count, dtype: int64,
 significant
 False    16
 Name: count, dtype: int64)

In [9]:
t1_advice_anova[t1_advice_anova['significant'] == True]

,model,feature,prompt_variant,stat,p_value,eta2,n_entailment,n_neutral,n_contradiction,significant,p_bh_corrected,_,alpha
0,gemma,clothing,default,7.734,0.000572,0.0677,15,171,30,True,0.004576,0.003201,0.003125
7,llama,resistance,default,9.064,0.000167,0.0784,39,63,114,True,0.002669,0.003201,0.003125
12,qwen,clothing,default,6.256,0.002290,0.0555,15,171,30,True,0.012213,0.003201,0.003125
13,qwen,victim_intoxication,default,4.341,0.014199,0.0392,18,156,42,True,0.045438,0.003201,0.003125
15,qwen,resistance,default,4.688,0.010176,0.0422,39,63,114,True,0.040706,0.003201,0.003125


In [10]:
dataset = "SurvivorStories"
t1_advice_anova = pd.read_csv(f'{analysis}/T1-ANOVA-{dataset}-advice.csv')
t1_summary_anova = pd.read_csv(f'{analysis}/T1-ANOVA-{dataset}-summary.csv')
t1_advice_anova['significant'].value_counts(), t1_summary_anova['significant'].value_counts()

(significant
 False    16
 Name: count, dtype: int64,
 significant
 False    16
 Name: count, dtype: int64)

## Tukey HSD

In [11]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pickle

with open('../../4_Evaluation-SetUp/MythSubspaces/SBERT.pkl', 'rb') as f:
    subspace = pickle.load(f)

narr_proj = pd.read_csv(f'../StatisticalTestResults/NarrativeProjections-{dataset}.csv')
NLI_LABELS = ["entailment", "neutral", "contradiction"]
MYTH_TYPES = ["clothing", "victim_intoxication", "perpetrator_intoxication", "resistance"]

significant = [
    ("gemma", "clothing"),
    ("llama", "resistance"),
    ("qwen", "clothing"),
    ("qwen", "victim_intoxication"),
    ("qwen", "resistance"),
]

for model, feat in significant:
    meta = pd.read_csv(f'../Results/{dataset}/1_Embeddings/{model}_advice_t1_metadata.csv')
    t1_embs = pd.read_pickle(f'../Results/{dataset}/1_Embeddings/SBERT/{model}_advice_t1.pkl')
    meta["proj_t1"] = t1_embs.apply(lambda v: np.dot(v, subspace)).values
    meta["narrative_idx"] = meta["narrative_idx"].astype(int)
    meta = meta.merge(narr_proj[["narrative_index"] + MYTH_TYPES],
                      left_on="narrative_idx", right_on="narrative_index", how="left")
    groups_data = meta.loc[meta[feat].isin(NLI_LABELS), ["proj_t1", feat]]
    tukey = pairwise_tukeyhsd(groups_data["proj_t1"], groups_data[feat])
    print(f"\n{model} — {feat}")
    print(tukey.summary())


gemma — clothing
     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
    group1      group2   meandiff p-adj   lower  upper  reject
--------------------------------------------------------------
contradiction entailment   0.0008 0.9989 -0.0434  0.045  False
contradiction    neutral   0.0067 0.8054 -0.0185 0.0318  False
   entailment    neutral   0.0058 0.9313 -0.0325 0.0442  False
--------------------------------------------------------------

llama — resistance
     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
    group1      group2   meandiff p-adj   lower  upper  reject
--------------------------------------------------------------
contradiction entailment   0.0114 0.3698 -0.0086 0.0313  False
contradiction    neutral   0.0143 0.2087 -0.0056 0.0343  False
   entailment    neutral   0.0029 0.9484 -0.0196 0.0255  False
--------------------------------------------------------------

qwen — clothing
     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
